# Understanding Memory in LLMs

In the previous Notebooks, we successfully explored how OpenAI models can enhance the results from Azure AI Search queries. 

However, we have yet to discover how to engage in a conversation with the LLM. With [Microsoft Copilot](http://chat.bing.com/), for example, this is possible, as it can understand and reference the previous responses.

There is a common misconception that LLMs (Large Language Models) have memory. This is not true. While they possess knowledge, they do not retain information from previous questions asked to them.

In this Notebook, our goal is to illustrate how we can effectively "endow the LLM with memory" by employing prompts and context.

In [1]:
import os
import random
from langchain_community.chat_message_histories import ChatMessageHistory, CosmosDBChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import ConfigurableFieldSpec
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from typing import List

from IPython.display import Markdown, HTML, display  

def printmd(string):
    display(Markdown(string))

#custom libraries that we will use later in the app
from common.utils import CustomAzureSearchRetriever, get_answer
from common.prompts import DOCSEARCH_PROMPT_TEXT

from dotenv import load_dotenv
load_dotenv("credentials.env")

import logging

# Get the root logger
logger = logging.getLogger()
# Set the logging level to a higher level to ignore INFO messages
logger.setLevel(logging.WARNING)

In [2]:
# Set the ENV variables that Langchain needs to connect to Azure OpenAI
os.environ["OPENAI_API_VERSION"] = os.environ["AZURE_OPENAI_API_VERSION"]

### Let's start with the basics
Let's use a very simple example to see if the GPT model of Azure OpenAI have memory. We again will be using langchain to simplify our code 

In [3]:
QUESTION = "tell me chinese medicines that help fight covid-19"
FOLLOW_UP_QUESTION = "What was my prior question?"

In [4]:
COMPLETION_TOKENS = 1000
# Create an OpenAI instance
llm = AzureChatOpenAI(deployment_name=os.environ["GPT4o_DEPLOYMENT_NAME"], 
                      temperature=0.5, max_tokens=COMPLETION_TOKENS)

In [5]:
# We create a very simple prompt template, just the question as is:
output_parser = StrOutputParser()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant that give thorough responses to users."),
    ("user", "{input}")
])

In [6]:
# Let's see what the GPT model responds
chain = prompt | llm | output_parser
response_to_initial_question = chain.invoke({"input": QUESTION})
display(Markdown(response_to_initial_question))

Traditional Chinese Medicine (TCM) has been used in China as a complementary approach to managing COVID-19 symptoms and supporting recovery. While TCM should not replace conventional medical treatments, it can be integrated into a comprehensive care plan under the guidance of qualified healthcare professionals. Here are some TCM formulations and herbs that have been explored for their potential benefits in the context of COVID-19:

1. **Lianhua Qingwen Capsules**: This is one of the most well-known TCM formulations used during the COVID-19 pandemic. It contains several herbs, including Forsythia suspensa and Lonicera japonica, and is traditionally used to clear heat and remove toxins.

2. **Jinhua Qinggan Granules**: This formulation is often used for relieving symptoms such as fever and cough. It includes herbs like Ephedra sinica and Mentha haplocalyx.

3. **Shufeng Jiedu Capsules**: These are used to relieve symptoms like sore throat and cough. The formulation includes herbs such as Radix Bupleuri and Radix Isatidis.

4. **Qingfei Paidu Decoction**: This is a classical formula that has been recommended for treating COVID-19 symptoms. It includes a combination of 21 herbs, such as Ephedra, Licorice, and Almond.

5. **Xuanfei Baidu Granules**: Developed to clear and detoxify the lungs, this formula includes a variety of herbs like Ephedra and Gypsum Fibrosum.

6. **Huashi Baidu Formula**: This is another herbal formula that has been used to manage COVID-19 symptoms, focusing on clearing dampness and detoxifying the body.

### Important Considerations:
- **Consultation**: Always consult with a healthcare provider or a qualified TCM practitioner before using any herbal remedies, especially if you have underlying health conditions or are taking other medications.
- **Quality and Safety**: Ensure that any TCM products you use are from reputable sources to avoid contamination or adulteration.
- **Complementary Role**: Remember that TCM should complement, not replace, conventional medical treatments like vaccines and antiviral medications.
- **Research and Evidence**: While some studies have shown potential benefits of TCM in managing COVID-19 symptoms, more research is needed to fully understand their efficacy and safety.

It's crucial to follow public health guidelines and rely on evidence-based treatments as the primary approach to managing COVID-19.

In [7]:
#Now let's ask a follow up question
printmd(chain.invoke({"input": FOLLOW_UP_QUESTION}))

I'm sorry, but I don't have access to previous interactions or any prior questions you may have asked. Each session is independent, and I don't have memory of past exchanges. If you have a question or need information, feel free to ask and I'll do my best to assist you!

As you can see, it doesn't remember what it just responded, sometimes it responds based only on the system prompt, or just randomly. This proof that the LLM does NOT have memory and that we need to give the memory as a a conversation history as part of the prompt, like this:

In [8]:
hist_prompt = ChatPromptTemplate.from_template(
"""
    {history}
    Human: {question}
    AI:
"""
)
chain = hist_prompt | llm | output_parser

In [9]:
Conversation_history = """
Human: {question}
AI: {response}
""".format(question=QUESTION, response=response_to_initial_question)

In [10]:
printmd(chain.invoke({"history":Conversation_history, "question": FOLLOW_UP_QUESTION}))

Your prior question was: "tell me chinese medicines that help fight covid-19."

**Bingo!**, so we now know how to create a chatbot using LLMs, we just need to keep the state/history of the conversation and pass it as context every time

## Now that we understand the concept of memory via adding history as a context, let's go back to our GPT Smart Search engine

From Langchain website:
    
A memory system needs to support two basic actions: reading and writing. Recall that every chain defines some core execution logic that expects certain inputs. Some of these inputs come directly from the user, but some of these inputs can come from memory. A chain will interact with its memory system twice in a given run.

    AFTER receiving the initial user inputs but BEFORE executing the core logic, a chain will READ from its memory system and augment the user inputs.
    AFTER executing the core logic but BEFORE returning the answer, a chain will WRITE the inputs and outputs of the current run to memory, so that they can be referred to in future runs.
    
So this process adds delays to the response, but it is a necessary delay :)

![image](./images/memory_diagram.png)

In [11]:
index1_name = "srch-index-files"
index2_name = "srch-index-csv"
index3_name = "srch-index-books"
indexes = [index1_name, index2_name, index3_name]

In [12]:
# Initialize our custom retriever 
retriever = CustomAzureSearchRetriever(indexes=indexes, topK=50, reranker_threshold=1)

**Prompt Template Definition**

If you check closely below, there is an optional variable in the `DOCSEARCH_PROMPT` called `history`. It is basically a placeholder were we will inject the conversation in the prompt so the LLM is aware of it before it answers.


In [13]:
DOCSEARCH_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", DOCSEARCH_PROMPT_TEXT + "\n\nCONTEXT:\n{context}\n\n"),
        MessagesPlaceholder(variable_name="history", optional=True),
        ("human", "{question}"),
    ]
)

printmd(DOCSEARCH_PROMPT.messages[0].prompt.template)



## On how to respond to humans based on Tool's retrieved information:
- Given extracted parts from one or multiple documents, and a question, answer the question thoroughly with citations/references. 
- In your answer, **You MUST use** all relevant extracted parts that are relevant to the question.
- **YOU MUST** place inline citations directly after the sentence they support using this Markdown format: `[[number]](url)`.
- The reference must be from the `source:` section of the extracted parts. You are not to make a reference from the content, only from the `source:` of the extract parts.
- Reference document's URL can include query parameters. Include these references in the document URL using this Markdown format: [[number]](url?query_parameters)
- **You must refuse** to provide any response if there is no relevant information in the conversation or on the retrieved documents.
- **You cannot add information to the context** from your pre-existing knowledge. You can only use the information on the retrieved documents, **NOTHING ELSE**.
- **Never** provide an answer without references to the retrieved content.
- Make sure the references provided are relevant and contains information that supports your answer. 
- You must refuse to provide any response if there is no relevant information from the retrieved documents. If no data is found, clearly state: 'The tools did not provide relevant information for this question. I cannot answer this from prior knowledge.' Repeat this process for any question that lacks relevant tool data.".
- If no information is retrieved, or if the retrieved information does not answer the question, you must refuse to answer and state clearly: 'The tools did not provide relevant information.'
- If multiple or conflicting explanations are present in the retrieved content, detail them all.




CONTEXT:
{context}



**Now let's add memory to it:**

In [14]:
store = {} # Our first memory will be a dictionary in memory

# We have to define a custom function that takes a session_id and looks somewhere
# (in this case in a dictionary in memory) for the conversation
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [15]:
# We use our original chain with the retriever but removing the StrOutputParser
chain = (
    {
        "context": itemgetter("question") | retriever, 
        "question": itemgetter("question"),
        "history": itemgetter("history")
    }
    | DOCSEARCH_PROMPT
    | llm
)

## Then we pass the above chain to another chain that adds memory to it

output_parser = StrOutputParser()

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
) | output_parser

In [16]:
# This is where we configure the session id
config={"configurable": {"session_id": "abc123"}}

In [17]:
printmd(chain_with_history.invoke({"question": QUESTION}, config=config))

Traditional Chinese Medicine (TCM) has been used in various ways to help combat COVID-19. Several treatments and prescriptions have been recommended and studied:

1. **Qingfei Paidu Decoction**: This has been recommended by the National Health Commission of the People's Republic of China for the treatment of COVID-19. TCM shows good clinical efficacy and potential in treating COVID-19, with previous studies indicating broad-spectrum antiviral activity [[7]](https://doi.org/10.19540/j.cnki.cjcmm.20200219.501; https://www.ncbi.nlm.nih.gov/pubmed/32281335/).

2. **Ma Xing Shi Gan Decoction (MXSGD)**: This decoction has been widely applied in the clinical treatment of COVID-19. It is believed to reduce inflammation, suppress cytokine storms, protect the pulmonary alveolar-capillary barrier, alleviate pulmonary edema, regulate the immune response, and decrease fever [[26]](https://doi.org/10.26355/eurrev_202003_20704; https://www.ncbi.nlm.nih.gov/pubmed/32271454/).

3. **Shuang Huang Lian Kou Fu Ye and other TCM combinations**: These have been used to treat patients infected by SARS-CoV-2 by inhibiting the virus's proliferation and triggering molecular pathways to fight the virus [[1]](https://doi.org/10.1101/2020.04.10.20060376).

4. **Herbal Formulas and Patent Medicines**: Various herbal formulas and patent medicines have been used based on the clinical classification and TCM pattern diagnoses. These include Yin Qiao Powder, Huopo Xialing Decoction, Maxing Shigan Decoction, and others, which focus on clearing heat, ventilating the lung, removing toxicity, and eliminating evil [[38]](https://www.ncbi.nlm.nih.gov/pubmed/32268018/).

5. **Integrated Medicine Protocols**: The clinical application of integrative medicine protocols, combining TCM and Western medicine, have shown advantages in improving patient symptoms, shortening the course of disease, delaying disease progression, and reducing mortality [[33]](https://doi.org/10.7501/j.issn.0253-2670.2020.04.008).

These treatments highlight the role of TCM in enhancing body resistance, eliminating pathogenic factors, and supporting the immune system during the COVID-19 pandemic. However, further clinical studies are necessary to verify their effectiveness comprehensively.

In [18]:
# Remembers
printmd(chain_with_history.invoke({"question": FOLLOW_UP_QUESTION},config=config))

Your prior question was about "chinese medicines that help fight covid-19."

In [19]:
# Remembers
printmd(chain_with_history.invoke({"question": "Thank you! Good bye"},config=config))

You're welcome! Goodbye! If you have more questions in the future, feel free to ask.

## Using CosmosDB as persistent memory

Previously, we  added local RAM memory to our chatbot. However, it is not persistent, it gets deleted once the app user's session is terminated. It is necessary then to use a Database for persistent storage of each of the  user conversations, not only for Analytics and Auditing, but also if we wish to provide recommendations in the future. 

In the next notebook we are going to explain how to use an external Database (CosmosDB) to keep the state of the conversation.

# Summary
##### Adding memory to our application allows the user to have a conversation, however this feature is not something that comes with the LLM, but instead, memory is something that we must provide to the LLM in form of context of the question.

We added persitent memory using local RAM.

We also can notice that the current chain that we are using is smart, but not that much. Although we have given memory to it, many times it searches for similar docs everytime, regardless of the input. This doesn't seem efficient, but regardless, we are very close to finish our first RAG talk-to-your-data bot.

Note:The use of `RunnableWithMessageHistory` in this notebook is for example purposes. We will see later (on the next notebooks), that we recomend the use of memory state and graphs in order to inject memory into an bot. 

# NEXT
We know now how to do a Smart Search Engine that can power a chatbot!! great!

In the next notebook 6, we are going to build our first RAG bot. In order to do this we will introduce the concept of Agents.

# Let's check the messages

In [33]:
i = 0
for m in store["abc123"].messages:
    i += 1
    print(f"message {i} - {m.content}\n")

message 1 - tell me chinese medicines that help fight covid-19

message 2 - Traditional Chinese Medicine (TCM) has been used in various ways to help combat COVID-19. Several treatments and prescriptions have been recommended and studied:

1. **Qingfei Paidu Decoction**: This has been recommended by the National Health Commission of the People's Republic of China for the treatment of COVID-19. TCM shows good clinical efficacy and potential in treating COVID-19, with previous studies indicating broad-spectrum antiviral activity [[7]](https://doi.org/10.19540/j.cnki.cjcmm.20200219.501; https://www.ncbi.nlm.nih.gov/pubmed/32281335/).

2. **Ma Xing Shi Gan Decoction (MXSGD)**: This decoction has been widely applied in the clinical treatment of COVID-19. It is believed to reduce inflammation, suppress cytokine storms, protect the pulmonary alveolar-capillary barrier, alleviate pulmonary edema, regulate the immune response, and decrease fever [[26]](https://doi.org/10.26355/eurrev_202003_2070